# 09 — kvpress: Qasper Benchmark with PrefillDecodingPress

This notebook evaluates KV cache compression on the
[Qasper](https://huggingface.co/datasets/tau/scrolls) benchmark (from SCROLLS)
using [kvpress](https://github.com/NVIDIA/kvpress) with Qwen3-8B.

Qasper tests document QA on full NLP research papers (~3K–8K tokens),
requiring both long-context comprehension (prefill) and answer generation
(decoding).

We test **KeyDiffPress**-based compression applied to both **prefill** and
**decoding** phases using PrefillDecodingPress, with two decoding strategies:
- **full_replacement** — CompressionRatioDecodingPress
- **filtering** — FilteringPress

Scoring uses the HuggingFace `evaluate` library (SQuAD F1).

Results are saved to `results/kvpress_qasper/` for comparison in later notebooks.

## Configuration

In [7]:
!pip install evaluate

In [1]:
import os
os.environ["HF_HOME"] = "/opt/app-root/src/.cache/huggingface"

MODEL_NAME = "Qwen/Qwen3-8B"

COMPRESSION_RATIOS = [0.01, 0.25, 0.50, 0.75]

FRACTION = 0.01

MAX_NEW_TOKENS = 64

PRESS_CONFIGS = {
    "full_replacement": lambda cr: PrefillDecodingPress(
        prefilling_press=KeyDiffPress(compression_ratio=cr),
        decoding_press=CompressionRatioDecodingPress(
            base_press=KeyDiffPress(), target_compression_ratio=cr,
        ),
    ),
    "filtering": lambda cr: PrefillDecodingPress(
        prefilling_press=KeyDiffPress(compression_ratio=cr),
        decoding_press=FilteringPress(
            base_press=KeyDiffPress(), target_compression_ratio=cr,
            fill_padding=False,
        ),
    ),
}

In [2]:
import sys
import builtins

_original_print = builtins.print

def print(*args, **kwargs):
    _original_print(*args, **kwargs)
    if sys.stdout is not sys.__stdout__:
        kwargs['file'] = sys.__stdout__
        kwargs['flush'] = True
        _original_print(*args, **kwargs)

In [3]:
import sys
import os

FORK_DIR = "/opt/app-root/src/kvpress-fork"

if os.path.isdir(FORK_DIR) and os.listdir(FORK_DIR):
    sys.path.insert(0, FORK_DIR)
    import kvpress
    print(f"Using FORK kvpress from {FORK_DIR}")
else:
    import kvpress
    print(f"Using SYSTEM kvpress")

print(f"  location: {os.path.dirname(kvpress.__file__)}")

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using FORK kvpress from /opt/app-root/src/kvpress-fork
Using FORK kvpress from /opt/app-root/src/kvpress-fork
  location: /opt/app-root/src/kvpress-fork/kvpress
  location: /opt/app-root/src/kvpress-fork/kvpress


## 1. Load Model

In [4]:
import torch
from transformers import pipeline
from kvpress import (
    KeyDiffPress, PrefillDecodingPress, CompressionRatioDecodingPress,
    FilteringPress,
)

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)} — {vram_gb:.1f} GB VRAM")

model_kwargs = {}

try:
    import flash_attn  # noqa: F401
    model_kwargs["attn_implementation"] = "flash_attention_2"
    print("Using Flash Attention 2")
except ImportError:
    print("Flash Attention 2 not available, using default attention")

pipe = pipeline(
    "kv-press-text-generation",
    model=MODEL_NAME,
    device_map="auto",
    model_kwargs=model_kwargs,
    trust_remote_code=True,
)

print(f"\nModel loaded. GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

GPU: NVIDIA A100-SXM4-40GB — 42.4 GB VRAM
Using Flash Attention 2
GPU: NVIDIA A100-SXM4-40GB — 42.4 GB VRAM
Using Flash Attention 2


Loading checkpoint shards: 100%|██████████| 5/5 [00:04<00:00,  1.10it/s]
Device set to use cuda:0



Model loaded. GPU memory allocated: 16.38 GB

Model loaded. GPU memory allocated: 16.38 GB


## 2. Load Qasper Dataset

In [5]:
from datasets import load_dataset

qasper_ds = load_dataset("tau/scrolls", "qasper", split="validation")
if FRACTION < 1.0:
    n = max(1, int(len(qasper_ds) * FRACTION))
    qasper_ds = qasper_ds.select(range(n))

print(f"Qasper dataset: {len(qasper_ds)} examples")
print(f"Columns: {qasper_ds.column_names}")
print(f"\nSample input (first 200 chars): {qasper_ds[0]['input'][:200]}...")
print(f"Sample output: {qasper_ds[0]['output']}")

The repository for tau/scrolls contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/tau/scrolls.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


Generating train split: 2567 examples [00:00, 5774.94 examples/s]
Generating validation split: 1726 examples [00:00, 6896.77 examples/s]
Generating test split: 3437 examples [00:00, 6948.28 examples/s]

Qasper dataset: 17 examples
Qasper dataset: 17 examplesColumns: ['id', 'pid', 'input', 'output']

Sample input (first 200 chars): which multilingual approaches do they compare with?

Introduction
Although Neural Machine Translation (NMT) has dominated recent research on translation tasks BIBREF0, BIBREF1, BIBREF2, NMT heavily re...
Sample output: BIBREF19, BIBREF20

Columns: ['id', 'pid', 'input', 'output']

Sample input (first 200 chars): which multilingual approaches do they compare with?

Introduction
Although Neural Machine Translation (NMT) has dominated recent research on translation tasks BIBREF0, BIBREF1, BIBREF2, NMT heavily re...
Sample output: BIBREF19, BIBREF20


## 3. Load Scoring Metric

In [8]:
import evaluate

squad_metric = evaluate.load("squad")
print("Loaded SQuAD metric (token-level F1)")

Loaded SQuAD metric (token-level F1)
Loaded SQuAD metric (token-level F1)


## 4. Run Inference

For each (algorithm, compression_ratio) combination, run all Qasper
examples through the kvpress pipeline. The SCROLLS `input` field
already contains the full paper text with the question appended.

In [9]:
import time

all_results = []

configs = [("no_press", 0.0, None)]
for press_name, press_factory in PRESS_CONFIGS.items():
    for ratio in COMPRESSION_RATIOS:
        configs.append((press_name, ratio, press_factory(ratio)))

for press_name, ratio, press in configs:
    label = f"{press_name} | ratio={ratio}"
    print(f"\n{'='*60}")
    print(f"Running: {label} ({len(qasper_ds)} examples)")
    print(f"{'='*60}")

    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    log_every = max(1, len(qasper_ds) // 10)

    for i, row in enumerate(qasper_ds):
        kwargs = dict(
            question="",
            answer_prefix="",
            max_new_tokens=MAX_NEW_TOKENS,
        )
        if press is not None:
            kwargs["press"] = press

        t_start = time.perf_counter()
        output = pipe(row["input"], **kwargs)
        elapsed = time.perf_counter() - t_start

        all_results.append({
            "framework": "kvpress",
            "press": press_name,
            "compression_ratio": ratio,
            "predicted_answer": output["answer"],
            "reference_answer": row["output"],
            "elapsed_sec": round(elapsed, 3),
        })

        if (i + 1) % log_every == 0 or (i + 1) == len(qasper_ds):
            total_elapsed = time.perf_counter() - t0
            print(f"  {i+1}/{len(qasper_ds)} — {total_elapsed:.0f}s elapsed")

    total_elapsed = time.perf_counter() - t0
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"  Done: {total_elapsed:.0f}s, peak_mem={peak_mem:.2f}GB")

    torch.cuda.empty_cache()

print(f"\nTotal results: {len(all_results)}")


Running: no_press | ratio=0.0 (17 examples)

Running: no_press | ratio=0.0 (17 examples)
  1/17 — 4s elapsed
  1/17 — 4s elapsed
  2/17 — 8s elapsed
  2/17 — 8s elapsed
  3/17 — 12s elapsed
  3/17 — 12s elapsed
  4/17 — 16s elapsed
  4/17 — 16s elapsed
  5/17 — 19s elapsed
  5/17 — 19s elapsed
  6/17 — 23s elapsed
  6/17 — 23s elapsed
  7/17 — 27s elapsed
  7/17 — 27s elapsed
  8/17 — 31s elapsed
  8/17 — 31s elapsed
  9/17 — 35s elapsed
  9/17 — 35s elapsed


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  10/17 — 38s elapsed
  10/17 — 38s elapsed
  11/17 — 42s elapsed
  11/17 — 42s elapsed
  12/17 — 46s elapsed
  12/17 — 46s elapsed
  13/17 — 49s elapsed
  13/17 — 49s elapsed
  14/17 — 53s elapsed
  14/17 — 53s elapsed
  15/17 — 57s elapsed
  15/17 — 57s elapsed
  16/17 — 61s elapsed
  16/17 — 61s elapsed
  17/17 — 65s elapsed
  Done: 65s, peak_mem=17.85GB
  17/17 — 65s elapsed
  Done: 65s, peak_mem=17.85GB

Running: full_replacement | ratio=0.01 (17 examples)

Running: full_replacement | ratio=0.01 (17 examples)
  1/17 — 6s elapsed
  1/17 — 6s elapsed
  2/17 — 10s elapsed
  2/17 — 10s elapsed
  3/17 — 14s elapsed
  3/17 — 14s elapsed
  4/17 — 18s elapsed
  4/17 — 18s elapsed
  5/17 — 23s elapsed
  5/17 — 23s elapsed
  6/17 — 27s elapsed
  6/17 — 27s elapsed
  7/17 — 31s elapsed
  7/17 — 31s elapsed
  8/17 — 35s elapsed
  8/17 — 35s elapsed
  9/17 — 40s elapsed
  9/17 — 40s elapsed
  10/17 — 44s elapsed
  10/17 — 44s elapsed
  11/17 — 48s elapsed
  11/17 — 48s elapsed
  12/17 — 52s el

## 5. Score & Results

Score predictions using the HuggingFace `evaluate` SQuAD metric
(token-level F1). SCROLLS references may be pipe-delimited for
multiple valid answers.

In [10]:
import pandas as pd

df = pd.DataFrame(all_results)

all_metrics = {}
rows = []
for (press, ratio), group in df.groupby(["press", "compression_ratio"]):
    predictions = [
        {"id": str(i), "prediction_text": row["predicted_answer"]}
        for i, (_, row) in enumerate(group.iterrows())
    ]
    references = [
        {
            "id": str(i),
            "answers": {
                "text": row["reference_answer"].split("|"),
                "answer_start": [0] * len(row["reference_answer"].split("|")),
            },
        }
        for i, (_, row) in enumerate(group.iterrows())
    ]

    result = squad_metric.compute(predictions=predictions, references=references)
    key = f"{press}__{ratio}"
    all_metrics[key] = result
    mean_time = group["elapsed_sec"].mean()
    rows.append({
        "press": press, "compression_ratio": ratio,
        "f1": round(result["f1"], 2),
        "exact_match": round(result["exact_match"], 2),
        "mean_time": round(mean_time, 3),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

           press  compression_ratio   f1  exact_match  mean_time
       filtering               0.01 8.56          0.0     11.934
       filtering               0.25 9.88          0.0     12.254
       filtering               0.50 8.27          0.0     12.205
       filtering               0.75 3.35          0.0     12.121
full_replacement               0.01 8.56          0.0      4.334
full_replacement               0.25 9.54          0.0      4.244
full_replacement               0.50 7.81          0.0      4.244
full_replacement               0.75 3.25          0.0      4.244
        no_press               0.00 7.62          0.0      3.804
           press  compression_ratio   f1  exact_match  mean_time
       filtering               0.01 8.56          0.0     11.934
       filtering               0.25 9.88          0.0     12.254
       filtering               0.50 8.27          0.0     12.205
       filtering               0.75 3.35          0.0     12.121
full_replacement         

## 6. Save Results

In [11]:
import json

os.makedirs("results/kvpress_qasper", exist_ok=True)

predictions_path = "results/kvpress_qasper/predictions.csv"
df.to_csv(predictions_path, index=False)
print(f"Saved predictions to {predictions_path}")

metrics_path = "results/kvpress_qasper/metrics.json"
with open(metrics_path, "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"Saved metrics to {metrics_path}")

Saved predictions to results/kvpress_qasper/predictions.csv
Saved predictions to results/kvpress_qasper/predictions.csv
Saved metrics to results/kvpress_qasper/metrics.json
Saved metrics to results/kvpress_qasper/metrics.json
